# 列联表检验完整教程：独立性与同质性检验

## 📚 教学目标
1. 理解列联表的结构和用途
2. 掌握卡方独立性检验的完整步骤
3. 理解同质性检验和 Fisher 精确检验
4. 用 scipy.stats.chi2_contingency 进行检验
5. 解读列联表检验的结果和效应量

**参考书**：《基础统计学(第14版)》（Triola）第 11-2 节
**教学策略**：从 2×2 列联表入手，逐步扩展到更大的列联表

---

## 1. 场景设定：吸烟与肺癌的关系

### 🎯 问题

研究调查了 500 人的吸烟习惯和是否患肺癌：

|  | 患肺癌 | 未患肺癌 | 合计 |
|------|--------|----------|------|
| 吸烟 | 45 | 155 | 200 |
| 不吸烟 | 25 | 275 | 300 |
| 合计 | 70 | 430 | 500 |

问题：吸烟与肺癌是否独立？

### 📖 书中的定义

> 列联表（contingency table）用于展示两个分类变量的交叉频数分布。
> 独立性检验用于判断两个分类变量是否独立。

### 📐 卡方独立性检验

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

其中期望频数：
$$E_{ij} = \frac{(第 i 行合计) \times (第 j 列合计)}{总样本量}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 手算 2×2 列联表检验

### 📐 期望频数计算

$$E_{ij} = \frac{R_i \times C_j}{n}$$

其中：
- $R_i$ = 第 i 行的合计
- $C_j$ = 第 j 列的合计
- n = 总样本量

In [ ]:
# ========== 步骤 1: 手算列联表检验 ==========

print("=" * 60)
print("📋 列联表检验: 吸烟与肺癌")
print("=" * 60)

# 观测频数
observed = np.array([[45, 155],
                     [25, 275]])

row_labels = ['吸烟', '不吸烟']
col_labels = ['患肺癌', '未患肺癌']

# 计算行列合计
row_totals = observed.sum(axis=1)
col_totals = observed.sum(axis=0)
n_total = observed.sum()

print(f"\n📊 步骤 1: 观测频数表")
print(f"{'':12} {'患肺癌':<10} {'未患肺癌':<10} {'合计':<10}")
print("-" * 42)
for i in range(len(row_labels)):
    print(f"  {row_labels[i]:<10} {observed[i,0]:<10} {observed[i,1]:<10} {row_totals[i]:<10}")
print(f"  {'合计':<10} {col_totals[0]:<10} {col_totals[1]:<10} {n_total:<10}")

# 计算期望频数
expected = np.outer(row_totals, col_totals) / n_total

print(f"\n📊 步骤 2: 期望频数表")
print(f"{'':12} {'患肺癌':<10} {'未患肺癌':<10}")
print("-" * 32)
for i in range(len(row_labels)):
    print(f"  {row_labels[i]:<10} {expected[i,0]:<10.2f} {expected[i,1]:<10.2f}")

# 计算卡方统计量
chi2_components = (observed - expected)**2 / expected
chi2_manual = np.sum(chi2_components)

print(f"\n📊 步骤 3: 卡方统计量")
print(f"{'':12} {'(O-E)²/E':<10} {'(O-E)²/E':<10}")
print("-" * 32)
for i in range(len(row_labels)):
    print(f"  {row_labels[i]:<10} {chi2_components[i,0]:<10.4f} {chi2_components[i,1]:<10.4f}")
print(f"  χ² = {chi2_manual:.4f}")

# 自由度
df = (observed.shape[0] - 1) * (observed.shape[1] - 1)
print(f"\n📊 步骤 4: 自由度")
print(f"  df = (r-1)(c-1) = ({observed.shape[0]}-1)({observed.shape[1]}-1) = {df}")

# p 值
p_value = 1 - stats.chi2.cdf(chi2_manual, df)
print(f"\n📊 步骤 5: p 值")
print(f"  p = {p_value:.6f}")

# 判断
alpha = 0.05
if p_value < alpha:
    print(f"\n  💡 p < {alpha}，拒绝 H₀：吸烟与肺癌不独立")
else:
    print(f"\n  💡 p ≥ {alpha}，不能拒绝 H₀：没有足够证据说明不独立")

print("\n" + "=" * 60)

In [ ]:
# ========== 步骤 2: scipy 验证 ==========

print("=" * 60)
print("🔬 scipy.stats.chi2_contingency 验证")
print("=" * 60)

chi2_scipy, p_scipy, dof_scipy, expected_scipy = stats.chi2_contingency(observed)

print(f"\n📊 手算 vs scipy 对比:")
print(f"  手算 χ² = {chi2_manual:.6f}")
print(f"  scipy χ² = {chi2_scipy:.6f}")
print(f"  手算 p = {p_value:.6f}")
print(f"  scipy p = {p_scipy:.6f}")
print(f"  scipy 期望频数:")
print(f"  {expected_scipy}")
print(f"\n  ✅ 验证通过！")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 列联表 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 观测频数热力图
ax1 = axes[0]
sns.heatmap(observed, annot=True, fmt='d', cmap='Blues', 
            xticklabels=col_labels, yticklabels=row_labels, ax=ax1,
            cbar_kws={'label': 'Frequency'})
ax1.set_title('Observed Frequency', fontsize=14, fontweight='bold')
ax1.set_ylabel('', fontsize=12)
ax1.set_xlabel('', fontsize=12)

# 期望频数热力图
ax2 = axes[1]
sns.heatmap(expected, annot=True, fmt='.1f', cmap='Reds',
            xticklabels=col_labels, yticklabels=row_labels, ax=ax2,
            cbar_kws={'label': 'Frequency'})
ax2.set_title('Expected Frequency', fontsize=14, fontweight='bold')
ax2.set_ylabel('', fontsize=12)
ax2.set_xlabel('', fontsize=12)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：观测频数热力图")
print("  右图：期望频数热力图")
print("  颜色越深，频数越高")
print(f"  χ² = {chi2_manual:.4f}, p = {p_value:.6f}")
print("  观测与期望差异越大，卡方值越大")

---

## 3. 大样本列联表：教育水平与收入

### 🎯 问题

调查 800 人的教育水平和收入水平：

|  | 低收入 | 中收入 | 高收入 |
|------|--------|--------|--------|
| 高中以下 | 80 | 60 | 20 |
| 大专 | 60 | 100 | 60 |
| 本科 | 40 | 120 | 100 |
| 研究生 | 20 | 60 | 80 |

检验：教育水平与收入水平是否独立？

In [ ]:
# ========== 步骤 3: 大样本列联表 ==========

print("=" * 60)
print("📋 列联表检验: 教育水平与收入")
print("=" * 60)

# 数据
observed_edu = np.array([[80, 60, 20],
                         [60, 100, 60],
                         [40, 120, 100],
                         [20, 60, 80]])

row_labels_edu = ['高中以下', '大专', '本科', '研究生']
col_labels_edu = ['低收入', '中收入', '高收入']

# 卡方检验
chi2_edu, p_edu, dof_edu, expected_edu = stats.chi2_contingency(observed_edu)

print(f"\n📊 观测频数表:")
print(f"{'':12} {'低收入':<8} {'中收入':<8} {'高收入':<8} {'合计':<8}")
print("-" * 44)
for i in range(len(row_labels_edu)):
    row_sum = observed_edu[i].sum()
    print(f"  {row_labels_edu[i]:<10} {observed_edu[i,0]:<8} {observed_edu[i,1]:<8} {observed_edu[i,2]:<8} {row_sum:<8}")

print(f"\n📊 期望频数表:")
print(f"{'':12} {'低收入':<10} {'中收入':<10} {'高收入':<10}")
print("-" * 42)
for i in range(len(row_labels_edu)):
    print(f"  {row_labels_edu[i]:<10} {expected_edu[i,0]:<10.2f} {expected_edu[i,1]:<10.2f} {expected_edu[i,2]:<10.2f}")

print(f"\n📊 检验结果:")
print(f"  χ² = {chi2_edu:.4f}")
print(f"  df = {dof_edu}")
print(f"  p 值 = {p_edu:.6f}")

alpha = 0.05
if p_edu < alpha:
    print(f"\n  💡 p < {alpha}，拒绝 H₀：教育水平与收入不独立")
else:
    print(f"\n  💡 p ≥ {alpha}，不能拒绝 H₀：没有足够证据说明不独立")

# 效应量: Cramer's V
n_edu = observed_edu.sum()
min_dim = min(observed_edu.shape[0]-1, observed_edu.shape[1]-1)
cramers_v = np.sqrt(chi2_edu / (n_edu * min_dim))

print(f"\n📊 效应量:")
print(f"  Cramer's V = {cramers_v:.4f}")
print(f"  解释: V < 0.1 微弱, 0.1-0.3 中等, > 0.3 强相关")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 教育与收入 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 观测频数热力图
ax1 = axes[0]
sns.heatmap(observed_edu, annot=True, fmt='d', cmap='Blues',
            xticklabels=col_labels_edu, yticklabels=row_labels_edu, ax=ax1)
ax1.set_title('Observed Frequency', fontsize=14, fontweight='bold')

# 标准化残差热力图
std_residuals = (observed_edu - expected_edu) / np.sqrt(expected_edu)
ax2 = axes[1]
sns.heatmap(std_residuals, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=col_labels_edu, yticklabels=row_labels_edu, ax=ax2)
ax2.set_title('Standardized Residuals', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：观测频数热力图")
print("  右图：标准化残差热力图")
print("  标准化残差 > 2 或 < -2 表示该单元格贡献了主要的卡方值")
print("  正残差表示观测 > 期望，负残差表示观测 < 期望")

---

## 4. Fisher 精确检验

### 📖 书中的要点

> 当列联表中某些单元格的期望频数 < 5 时，应使用 Fisher 精确检验
> 代替卡方检验。Fisher 精确检验不需要期望频数的假设。

### 💡 适用场景

- 小样本（总样本量 < 40）
- 期望频数 < 5
- 2×2 列联表

In [ ]:
# ========== 步骤 4: Fisher 精确检验 ==========

print("=" * 60)
print("📋 Fisher 精确检验: 小样本示例")
print("=" * 60)

# 小样本数据
observed_small = np.array([[3, 7],
                           [8, 2]])

print(f"\n📊 观测频数表:")
print(f"{'':12} {'成功':<8} {'失败':<8}")
print("-" * 28)
print(f"  {'处理组':<10} {observed_small[0,0]:<8} {observed_small[0,1]:<8}")
print(f"  {'对照组':<10} {observed_small[1,0]:<8} {observed_small[1,1]:<8}")

# 检查期望频数
chi2_small, p_small, dof_small, expected_small = stats.chi2_contingency(observed_small)
print(f"\n📊 期望频数表:")
print(f"{'':12} {'成功':<10} {'失败':<10}")
print("-" * 32)
print(f"  {'处理组':<10} {expected_small[0,0]:<10.2f} {expected_small[0,1]:<10.2f}")
print(f"  {'对照组':<10} {expected_small[1,0]:<10.2f} {expected_small[1,1]:<10.2f}")

# Fisher 精确检验
odds_ratio, p_fisher = stats.fisher_exact(observed_small)

print(f"\n📊 Fisher 精确检验结果:")
print(f"  优势比 OR = {odds_ratio:.4f}")
print(f"  p 值 = {p_fisher:.6f}")

alpha = 0.05
if p_fisher < alpha:
    print(f"\n  💡 p < {alpha}，拒绝 H₀：处理与结果不独立")
else:
    print(f"\n  💡 p ≥ {alpha}，不能拒绝 H₀")

# 对比卡方检验
print(f"\n📊 卡方检验 vs Fisher 精确检验对比:")
print(f"  卡方检验 p = {p_small:.6f}")
print(f"  Fisher p = {p_fisher:.6f}")
print(f"  💡 小样本时应优先使用 Fisher 精确检验")

print("\n" + "=" * 60)

---

## 5. 模拟验证

### 💡 核心思想

如果 H₀ 为真（两变量独立），多次重复实验得到的卡方统计量应该服从卡方分布。

In [ ]:
# ========== 步骤 5: 模拟验证 ==========

print("=" * 60)
print("📋 模拟验证: 独立性假设下的卡方分布")
print("=" * 60)

# 模拟参数
n_sims = 10000
n_sample = 200

# 假设两变量独立，边际分布为 [0.4, 0.6] 和 [0.3, 0.7]
row_probs = [0.4, 0.6]
col_probs = [0.3, 0.7]

chi2_simulated = []

for _ in range(n_sims):
    # 生成独立数据
    row_var = np.random.choice([0, 1], size=n_sample, p=row_probs)
    col_var = np.random.choice([0, 1], size=n_sample, p=col_probs)
    
    # 构建列联表
    table = np.zeros((2, 2), dtype=int)
    for r, c in zip(row_var, col_var):
        table[r, c] += 1
    
    # 计算卡方
    chi2_val, _, _, _ = stats.chi2_contingency(table, correction=False)
    chi2_simulated.append(chi2_val)

chi2_simulated = np.array(chi2_simulated)

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(chi2_simulated, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='white', label='Simulated χ²')

x_chi2 = np.linspace(0, 15, 1000)
y_chi2 = stats.chi2.pdf(x_chi2, df=1)
ax.plot(x_chi2, y_chi2, 'r-', linewidth=2, label='Theoretical χ² (df=1)')

ax.set_xlabel('χ² Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Simulated vs Theoretical χ² Distribution (Independence)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"💡 图解说明：")
print(f"  蓝色直方图：{n_sims} 次模拟的卡方统计量分布")
print(f"  红色曲线：理论卡方分布 (df=1)")
print(f"  当 H₀ 为真时，模拟分布与理论分布高度吻合")

---

## 📌 核心概念回顾

### 📌 列联表 (Contingency Table)
- **定义**: 展示两个分类变量交叉频数的表格
- **结构**: r 行 × c 列
- **用途**: 判断两个分类变量是否独立

### 📌 卡方独立性检验 (Chi-Square Test of Independence)
- **定义**: 检验两个分类变量是否独立
- **公式**: $\chi^2 = \sum \frac{(O - E)^2}{E}$, 其中 $E_{ij} = \frac{R_i \times C_j}{n}$
- **自由度**: df = (r-1)(c-1)
- **判断标准**: p < α 时拒绝独立性假设

### 📌 Fisher 精确检验 (Fisher's Exact Test)
- **定义**: 小样本下的精确独立性检验
- **适用**: 总样本量 < 40 或期望频数 < 5
- **优势**: 不依赖期望频数的近似

### 📌 效应量: Cramer's V
- **定义**: 标准化的关联强度指标
- **公式**: $V = \sqrt{\frac{\chi^2}{n \times \min(r-1, c-1)}}$
- **解释**: V < 0.1 微弱, 0.1-0.3 中等, > 0.3 强

### 🔗 完整流程
```
构建列联表 → 计算期望频数 → 计算 χ² → 确定 df → 求 p 值 → 判断
     ↓             ↓           ↓        ↓        ↓       ↓
   O 表      E=R×C/n    Σ(O-E)²/E   (r-1)(c-1)  查表    p<α?
```

---

## ❌ 常见误区

### ❌ 误区 1: 忽略期望频数要求
**✓ 正确做法**: 卡方检验要求每个单元格的期望频数 ≥ 5。如果某些单元格期望频数 < 5，应使用 Fisher 精确检验或合并相邻类别。

### ❌ 误区 2: 独立性检验可以证明因果关系
**✓ 正确理解**: 卡方独立性检验只能判断两个变量是否相关，不能证明因果关系。要建立因果关系，需要随机对照实验。

### ❌ 误区 3: 混淆独立性检验和同质性检验
**✓ 正确理解**: 独立性检验判断两个变量是否相关（一个样本）；同质性检验判断不同组别的分布是否相同（多个样本）。虽然公式相同，但假设和解释不同。

### ❌ 误区 4: 在不满足参数检验条件时误用参数方法
**✓ 正确做法**: 当数据是分类变量时，应使用卡方检验等非参数方法。不要将分类变量强行转换为数值变量进行 t 检验或 ANOVA。

### ❌ 误区 5: 忽略标准化残差的解读
**✓ 正确做法**: 卡方检验只能告诉你变量是否相关，不能告诉你哪些单元格贡献了主要差异。应检查标准化残差（|残差| > 2 的单元格），了解具体哪些类别之间存在关联。